# FG-7 code comments verification

Timestamp: 2026-09-13 14:51:16 +0400

## Intent

Add short beginner-friendly CSS, HTML, and JavaScript comments without changing application behavior or non-comment content.

## Method

Compare `index.html` with `HEAD:index.html` after stripping comments, syntax-check the extracted inline JavaScript, run a mock-DOM behavior test, and assert comment and CSS contracts.

In [1]:
import subprocess
result = subprocess.run(["git", "status", "--short", "--branch"], text=True, capture_output=True, check=True)
print(result.stdout, end="")
print(subprocess.run(["git", "log", "--oneline", "-2"], text=True, capture_output=True, check=True).stdout, end="")


## main...origin/main
 M index.html
?? experiments/2026-09-13-1451-code-comments.ipynb
502ddd3 Build random lunch menu generator
c50c802 Initial commit


In [2]:
from pathlib import Path
import re, subprocess
before = subprocess.run(["git", "show", "HEAD:index.html"], check=True, capture_output=True, text=True).stdout
after = Path("index.html").read_text()
def strip_comments(text):
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)
    text = re.sub(r"^\s*//[^\n]*(?:\n|$)", "", text, flags=re.M)
    return "".join(line for line in text.splitlines(keepends=True) if line.strip())
assert strip_comments(after) == strip_comments(before)
counts = (len(re.findall(r"/\*.*?\*/", after, re.S)), len(re.findall(r"<!--.*?-->", after, re.S)), len(re.findall(r"^\s*//[^\n]*", after, re.M)))
assert counts == (4, 2, 11)
assert "// Add click event to button." in after
print("non-comment equivalence: PASS")
print(f"comment syntax/counts: PASS (CSS={counts[0]}, HTML={counts[1]}, JS={counts[2]})")


non-comment equivalence: PASS
comment syntax/counts: PASS (CSS=4, HTML=2, JS=11)


In [3]:
import re, subprocess, tempfile
script = re.search(r"<script>([\s\S]*?)</script>", Path("index.html").read_text()).group(1)
with tempfile.NamedTemporaryFile("w", suffix=".js") as handle:
    handle.write(script); handle.flush()
    check = subprocess.run(["node", "--check", handle.name], text=True, capture_output=True)
assert check.returncode == 0, check.stderr
print("inline JavaScript node --check: PASS")


inline JavaScript node --check: PASS


In [4]:
import subprocess
behavior_check = r'''const assert = require("assert");
const fs = require("fs");
const vm = require("vm");
const html = fs.readFileSync("index.html", "utf8");
const script = html.match(/<script>([\s\S]*?)<\/script>/)[1];

class ClassList {
  constructor(initial = "") { this.values = new Set(initial.split(/\s+/).filter(Boolean)); }
  add(name) { this.values.add(name); }
  remove(name) { this.values.delete(name); }
  contains(name) { return this.values.has(name); }
}
class Element {
  constructor(id, classes = "") {
    this.id = id;
    this.value = "";
    this.textContent = "";
    this.href = "";
    this.src = "";
    this.alt = "";
    this.hidden = false;
    this.classList = new ClassList(classes);
    this.attributes = new Map();
    this.handlers = {};
    this.offsetWidth = 100;
  }
  addEventListener(type, handler) { this.handlers[type] = handler; }
  removeAttribute(name) { this.attributes.delete(name); }
}
const elements = {
  "#generate-button": new Element("generate-button", "generate-button"),
  "#cuisine-filter": new Element("cuisine-filter", "cuisine-filter"),
  "#result": new Element("result", "result"),
  "#food-photo": new Element("food-photo", "food-photo"),
  "#photo-placeholder": new Element("photo-placeholder", "photo-placeholder"),
  "#photo-credit": new Element("photo-credit", "photo-credit"),
  "#detail-cuisine": new Element("detail-cuisine"),
  "#detail-time": new Element("detail-time"),
  "#detail-difficulty": new Element("detail-difficulty"),
  "#recipe-link": new Element("recipe-link", "recipe-link is-hidden")
};
const sandbox = {
  document: { querySelector: selector => elements[selector] },
  Math: Object.create(Math),
  encodeURIComponent,
  console
};
let randomStep = 0;
sandbox.Math.random = () => ((randomStep++ * 37) % 997) / 997;
vm.createContext(sandbox);
vm.runInContext(script + "\nglobalThis.__dishes = lunchOptions;", sandbox);
const dishes = sandbox.__dishes;
assert.strictEqual(dishes.length, 20);
const expectedCuisines = ["American", "Asian", "Italian", "Japanese", "Mediterranean", "Mexican"];
assert.deepStrictEqual([...new Set(dishes.map(d => d.cuisine))].sort(), expectedCuisines);
const button = elements["#generate-button"];
for (const cuisine of ["All cuisines", ...expectedCuisines]) {
  elements["#cuisine-filter"].value = cuisine;
  let previous = null;
  for (let click = 0; click < 210; click++) {
    button.handlers.click();
    const name = elements["#result"].textContent.match(/^Today's pick: (.+)!\u00a0🥳$/u)[1];
    const dish = dishes.find(candidate => candidate.name === name);
    assert(dish);
    assert.notStrictEqual(name, previous);
    if (cuisine !== "All cuisines") assert.strictEqual(dish.cuisine, cuisine);
    assert.strictEqual(elements["#detail-cuisine"].textContent, dish.cuisine);
    assert.strictEqual(elements["#detail-time"].textContent, dish.time);
    assert.strictEqual(elements["#detail-difficulty"].textContent, dish.difficulty);
    assert.strictEqual(elements["#recipe-link"].textContent, "Find a recipe");
    assert.strictEqual(elements["#recipe-link"].href, `https://www.allrecipes.com/search?q=${encodeURIComponent(name)}`);
    assert(!elements["#recipe-link"].classList.contains("is-hidden"));
    assert.strictEqual(elements["#food-photo"].src, dish.photo);
    assert.strictEqual(elements["#food-photo"].alt, `A plate of ${name}`);
    assert.strictEqual(elements["#photo-placeholder"].hidden, true);
    assert(elements["#food-photo"].classList.contains("is-visible"));
    assert(elements["#photo-credit"].classList.contains("is-visible"));
    assert(elements["#result"].classList.contains("is-new"));
    assert(elements["#result"].textContent.includes("\u00a0"));
    previous = name;
  }
}
const compact = html.replace(/\s+/g, " ");
assert(/\.result\s*\{[^}]*height:\s*2\.5em;[^}]*min-height:\s*58px;/s.test(html));
assert(/\.dish-details\s*\{[^}]*min-height:\s*142px;/s.test(html));
assert(/\.food-photo\s*\{[^}]*position:\s*absolute;[^}]*inset:\s*0;[^}]*width:\s*100%;[^}]*height:\s*100%;[^}]*object-fit:\s*cover;[^}]*object-position:\s*center;/s.test(html));
console.log("behavior: PASS (20 dishes; 7 filters x 210 clicks; no consecutive repeats)");
console.log("updates: PASS (text/details/NBSP/recipe URL/photo/animation)");
console.log("CSS contracts: PASS (result/details heights; centered cover crop)");
'''
run = subprocess.run(["node", "-e", behavior_check], text=True, capture_output=True)
assert run.returncode == 0, run.stderr
print(run.stdout, end="")


behavior: PASS (20 dishes; 7 filters x 210 clicks; no consecutive repeats)
updates: PASS (text/details/NBSP/recipe URL/photo/animation)
CSS contracts: PASS (result/details heights; centered cover crop)


## Interpretation

All checks passed: comments use the required language-specific syntax, non-comment content remains equivalent to the clean base, inline JavaScript parses, and the mock-DOM checks preserve filtering, repeat prevention, updates, links, images, animation, NBSP, and CSS stability contracts.